# Read PtyRAD Output hdf5

- Created with PtyRAD v1.0.0
- Documentation: https://ptyrad.readthedocs.io/en/latest/
- PtyRAD paper: https://doi.org/10.1093/mam/ozaf070
- PtyRAD arXiv: https://arxiv.org/abs/2505.07814
- Zenodo record: https://doi.org/10.5281/zenodo.15273176
- Box folder: https://cornell.box.com/s/n5balzf88jixescp9l15ojx7di4xn1uo
- Youtube channel: https://www.youtube.com/@ptyrad_official

**Before running this notebook, you must first follow the instruction in `README.md` to:**
1. Create the Python environment with all dependant Python packages like PyTorch
2. Activate that python environment
3. Install `ptyrad` package into your activated Python environement (only need to install once)

> Note: This notebook introduces the internal structure of the PtyRAD HDF5 output and demonstrates how to extract and visualize reconstruction results using the `ptyrad.analysis.Analyzer` class.

Author: Chia-Hao Lee, cl2696@cornell.edu

## 00. Setup working directory and imports

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from ptyrad.analysis import Analyzer

# Change this to the ABSOLUTE PATH to the ptyrad/ folder so you can correctly access data/ and params/
work_dir = "/home/cl2696/altas/workspace/ptyrad"

os.chdir(work_dir)
print("Current working dir: ", os.getcwd())
# The printed working dir should be ".../ptyrad/" to locate the example params files easily

## 01. Load the output HDF5

`Analyzer` accepts a path to an `.hdf5` or `.h5` file and loads it into a nested Python `dict` internally. Construction is lightweight — no forward model is rebuilt unless you explicitly call `build_model()`.

In [ ]:
model_path = "output/tBL_WSe2/20260606_full_N16384_dp128_flipT100_random32_p6_1obj_6slice_dz2_orblur0.4_ozblur1.0_oposc_sng1.0_spr0.1/model_iter0200.hdf5"

analyzer = Analyzer(model_path)
print(analyzer)

### 01.A Internal structure — it's just a nested dict

The HDF5 is loaded as a plain nested Python `dict` accessible via `analyzer.data`. Every HDF5 group becomes a nested `dict`, and every dataset becomes a NumPy array (or scalar / string). You can inspect the full key hierarchy in dotted notation with `analyzer.get_keys()`.

In [ ]:
print("Top-level keys:")
for k in analyzer.data.keys():
    print(" ", k)

In [ ]:
# Full dotted-key listing of every array stored in the HDF5
print("\n".join(analyzer.get_keys()))

### 01.B Key sub-dicts

- `model_attributes` — propagator metadata: real-space pixel size (`dx`), reciprocal-space pixel size (`dk`), wavelength (`lambd`), integer crop positions (`crop_pos`), and other forward-model parameters
- `optimizable_tensors` — the AD-optimized arrays: `obja`, `objp`, `probe`, `probe_pos_shifts`, `obj_tilts`, `slice_thickness`
- `params` — a verbatim copy of the PtyRAD params used for this run
- `loss_iters`, `dz_iters`, `lr_iters`, `avg_tilt_iters`, `convergence_iters` — per-iteration training history

In [ ]:
print(f'model_attributes:\n  {list(analyzer.model_attrs.keys())}\n')
print(f'optimizable_tensors:\n  {list(analyzer.opt_tensors.keys())}\n')
print(f'params:\n  {list(analyzer.params.keys())}\n')

In [ ]:
# Convenient scalar properties derived from model_attributes and optimizable_tensors
print(f"Real-space pixel size  dx    = {analyzer.dx:.4f} Å")
print(f"Probe modes  pmode           = {analyzer.pmode}")
print(f"Object modes omode           = {analyzer.omode}")
print(f"Z-slices     nslice          = {analyzer.nslice}")
print(f"Multislice?                  = {analyzer.is_multislice}")
print(f"Saved at iteration           = {analyzer.niter}")

## 02. Convergence analysis

Before examining the reconstructed arrays, check the convergence diagnostics. The loss curve shows the total photon-counting loss summed over all scan positions; it should decrease and plateau. For multislice reconstructions, the slice-thickness evolution (`dz_iters`) shows how individual layer spacings were refined during optimization. The full convergence dashboard combines all monitored quantities in one view.

In [ ]:
analyzer.plot_loss()

In [ ]:
# Slice thickness evolution — only meaningful for multislice reconstructions (nslice > 1)
analyzer.plot_slice_thickness()

In [ ]:
analyzer.plot_dashboard()

## 03. Reconstructed object

The object is stored as amplitude (`obja`) and phase (`objp`) arrays of shape `(omode, Nz, Ny, Nx)`:
- **omode** — number of object modes (usually 1)
- **Nz** — number of z-slices (1 for single-slice, >1 for multislice)
- **Ny, Nx** — object canvas size, padded beyond the scanned FOV to keep the probe fully inside

Use `get_object_amplitude()` / `get_object_phase()` to retrieve the arrays. Pass `fov='crop'` to automatically clip to the scanned field of view.

In [ ]:
obja = analyzer.get_object_amplitude(fov='crop')
objp = analyzer.get_object_phase(fov='crop')
print("Object shape (omode, Nz, Ny, Nx):", obja.shape)

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(12, 6))
im0 = axs[0].imshow(obja.sum(0).prod(0))
im1 = axs[1].imshow(objp.sum(0).sum(0))
fig.colorbar(im0, shrink=0.6)
fig.colorbar(im1, shrink=0.6)
axs[0].set_title('Object amplitude (omode sum, z product)')
axs[1].set_title('Object phase (omode sum, z sum)')
plt.show()
# Amplitude is multiplicative along depth (transmission product); phase is additive (phase sum).

## 04. Reconstructed probe

The probe is stored as a complex-valued real-space wavefunction of shape `(pmode, Ny, Nx)`. Multiple probe modes capture incoherent illumination. `plot_probe()` can display real-space or Fourier-space amplitude/phase for each mode.

In [ ]:
probe = analyzer.get_probe(space='real')
print("Probe shape (pmode, Ny, Nx):", probe.shape)

In [ ]:
analyzer.plot_probe(space='real',    amp_or_phase='amplitude')
analyzer.plot_probe(space='real',    amp_or_phase='phase')
analyzer.plot_probe(space='fourier', amp_or_phase='amplitude')
analyzer.plot_probe(space='fourier', amp_or_phase='phase')

## 05. Scan positions

`crop_pos` contains the integer top-left corners of the probe windows on the object canvas (shape `(N, 2)`). PtyRAD refines sub-pixel offsets (`probe_pos_shifts`) during reconstruction. `get_probe_positions()` combines both and returns probe-center coordinates. You can retrieve positions in object pixels or physical Ångström units.

In [ ]:
pos = analyzer.get_probe_positions(fov='crop', target='probe', units='px')
print("Positions shape (N_scan, 2) [y, x]:", pos.shape)

In [ ]:
analyzer.plot_positions(fov='crop', target='probe')

In [ ]:
from ptyrad.plotting import plot_scan_positions

objp_crop = analyzer.get_object_phase(fov='crop').sum(0).sum(0)
pos_crop  = analyzer.get_probe_positions(fov='crop', target='probe', units='px')
plot_scan_positions(pos_crop, img=objp_crop, offset=(0, 0))
# Scatter points denote probe-center positions overlaid on the cropped object phase